In [3]:
import torch
from PIL import Image
from diffusers import AutoPipelineForText2Image, DDIMScheduler
from diffusers.utils import load_image

device = "cuda:3" if torch.cuda.is_available() else "cpu"

# Load a pipeline
pipeline = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to(device)

# Load the IP-Adapter weights
pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

models/ip-adapter_sd15.bin:   0%|          | 0.00/44.6M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

models/image_encoder/model.safetensors:   0%|          | 0.00/2.53G [00:00<?, ?B/s]

In [4]:
from diffusers.utils import load_image
image = load_image("/home1/koustav/Image_to_Image_Diffusion/output.png")

In [8]:
image_embeds = pipeline.prepare_ip_adapter_image_embeds(
    ip_adapter_image=image,
    ip_adapter_image_embeds=None,
    device=device,
    num_images_per_prompt=1,
    do_classifier_free_guidance=True,
)

print("Embedding tensor shape:", len(image_embeds))

Embedding tensor shape: 1


In [9]:
image_embeds[0].shape

torch.Size([2, 1, 1024])

In [ ]:
pipeline

StableDiffusionPipeline {
  "_class_name": "StableDiffusionPipeline",
  "_diffusers_version": "0.35.1",
  "_name_or_path": "runwayml/stable-diffusion-v1-5",
  "feature_extractor": [
    "transformers",
    "CLIPImageProcessor"
  ],
  "image_encoder": [
    "transformers",
    "CLIPVisionModelWithProjection"
  ],
  "requires_safety_checker": true,
  "safety_checker": [
    "stable_diffusion",
    "StableDiffusionSafetyChecker"
  ],
  "scheduler": [
    "diffusers",
    "PNDMScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

: 

In [ ]:
# Compute IP-Adapter embeddings for a sample image and save them
from diffusers.utils import load_image
import torch

# load an image (adjust path if needed)
image = load_image("/home1/koustav/Image_to_Image_Diffusion/output.png")
print("Loaded image size:", image.size)

# Use the pipeline helper to prepare IP-Adapter image embeddings
image_embeds = pipeline.prepare_ip_adapter_image_embeds(
    ip_adapter_image=image,
    ip_adapter_image_embeds=None,
    device=device,
    num_images_per_prompt=1,
    do_classifier_free_guidance=True,
)

print("Number of embed tensors:", len(image_embeds))
for i, e in enumerate(image_embeds):
    try:
        print(f"embed[{i}] shape: {tuple(e.shape)}, dtype={e.dtype}, device={e.device}")
    except Exception:
        print(f"embed[{i}] type: {type(e)}")

# Optionally save embeddings for later comparison
out_path = "ipadapter_embeds.pt"
torch.save(image_embeds, out_path)
print(f"Saved embeddings to {out_path}")
